<div dir="rtl">

# 💾 04 - CacheBackedEmbeddings in LangChain

## ما هو `CacheBackedEmbeddings`؟
- إعادة حساب الـ Embeddings لنفس النصوص يستهلك وقتاً وتكلفة مالية.
- `CacheBackedEmbeddings` تقوم بحفظ المتجهات المحسوبة في كاش محلي (`LocalFileStore`) أو في الذاكرة (`InMemoryByteStore`).
- عند طلب تضمين نص محسوب سابقاً، يُسترجع فوراً دون إعادة تمريره للنموذج.

</div>


In [ ]:
import time
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings

hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# إعداد مخزن الكاش المحلي
CACHE_DIR = Path("./.embeddings_cache")
file_store = LocalFileStore(str(CACHE_DIR))

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=hf_embeddings,
    document_embedding_cache=file_store,
    namespace=hf_embeddings.model_name
)

sample_docs = [
    "LangChain accelerates LLM application development.",
    "Vector stores enable semantic retrieval.",
    "Document loaders ingest unstructured data."
]

# 1. التشغيل الأول: حساب + تخزين في الكاش
t1 = time.time()
first_run = cached_embedder.embed_documents(sample_docs)
dur1 = time.time() - t1
print(f"⏱️ زمن التشغيل الأول (حساب جديد + كاش): {dur1:.5f} ثانية")

# 2. التشغيل الثاني: جلب فوري من الكاش
t2 = time.time()
second_run = cached_embedder.embed_documents(sample_docs)
dur2 = time.time() - t2
print(f"⚡ زمن التشغيل الثاني (استرجاع من الكاش): {dur2:.5f} ثانية")
print(f"🚀 نسبة التسريع: {dur1 / max(dur2, 1e-6):.1f}x أسرع!")
